# A/B Testing Tool: Statistical Significance in Python

This notebook calculates statistical significance for four conversion metrics:

- `add_payment_info / session`
- `add_shipping_info / session`
- `begin_checkout / session`
- `new_account / session`

The main calculation is performed in total for each A/B test. At the end of the notebook, there is an additional breakdown analysis by country, device, continent, and channel.

**Tableau dashboard access via the [link](https://public.tableau.com/app/profile/oleksandra.yakovenko/viz/ABTestingToolwithSignificance/Dashboard3)**

## 1. SQL query


In [1]:
query = """WITH
  session_info AS (
    SELECT
      s.date,
      s.ga_session_id,
      sp.country,
      sp.device,
      sp.continent,
      sp.channel,
      ab.test,
      ab.test_group
    FROM `data-analytics-mate.DA.ab_test` ab
    JOIN `data-analytics-mate.DA.session` s
      ON ab.ga_session_id = s.ga_session_id
    JOIN `data-analytics-mate.DA.session_params` sp
      ON sp.ga_session_id = ab.ga_session_id
  ),

  session_with_orders AS (
    SELECT
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      COUNT(DISTINCT o.ga_session_id) AS session_with_orders
    FROM `data-analytics-mate.DA.order` o
    JOIN session_info
      ON o.ga_session_id = session_info.ga_session_id
    GROUP BY
      session_info.date, session_info.country, session_info.device,
      session_info.continent, session_info.channel, session_info.test,
      session_info.test_group
  ),

  events AS (
    SELECT
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      ep.event_name,
      COUNT(DISTINCT ep.ga_session_id) AS event_cnt
    FROM `data-analytics-mate.DA.event_params` ep
    JOIN session_info
      ON ep.ga_session_id = session_info.ga_session_id
    GROUP BY
      session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      ep.event_name
  ),

session as (
  Select session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      count(distinct session_info.ga_session_id) as session_cnt
  from session_info
  group by session_info.date,
      session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group
),

account as (
  select session_info.date, session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group,
      count(distinct acs.ga_session_id) as new_account_cnt
  from `data-analytics-mate.DA.account_session` acs
  join session_info
  on acs.ga_session_id = session_info.ga_session_id
  group by session_info.date, session_info.country,
      session_info.device,
      session_info.continent,
      session_info.channel,
      session_info.test,
      session_info.test_group
)



select  session_with_orders.date,
      session_with_orders.country,
      session_with_orders.device,
      session_with_orders.continent,
      session_with_orders.channel,
      session_with_orders.test,
      session_with_orders.test_group,
      'session_with_orders' as event_name,
      session_with_orders.session_with_orders as value
from session_with_orders
union all
select events.date,
      events.country,
      events.device,
      events.continent,
      events.channel,
      events.test,
      events.test_group,
      event_name,
      event_cnt as value
from events
union all
select session.date,
      session.country,
      session.device,
      session.continent,
      session.channel,
      session.test,
      session.test_group,
      'session' as event_name,
      session_cnt as value
from session
union all
select account.date,
      account.country,
      account.device,
      account.continent,
      account.channel,
      account.test,
      account.test_group,
      'new_account' as event_name,
      new_account_cnt as value
from account"""

## 2. Importing libraries and loading data from BigQuery

In [2]:
!pip -q install statsmodels openpyxl

import numpy as np
import pandas as pd
from google.colab import auth, files
auth.authenticate_user()
project_id = 'data-analytics-mate'
from google.cloud import bigquery

client = bigquery.Client(project=project_id)

from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.multitest import multipletests

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")

query_job = client.query(query)
df = query_job.to_dataframe()

In [3]:
df.head()

,date,country,device,continent,channel,test,test_group,event_name,value
0,2020-12-08,Palestine,desktop,Asia,Direct,4,2,new_account,1
1,2020-12-08,Palestine,desktop,Asia,Direct,3,2,new_account,1
2,2020-11-06,Puerto Rico,desktop,Americas,Social Search,2,2,new_account,1
3,2020-11-06,Puerto Rico,desktop,Americas,Social Search,1,1,new_account,1
4,2020-12-08,Croatia,desktop,Europe,Direct,4,2,new_account,1


## 3. Data validation

The dataset must contain the following columns:

`date`, `country`, `device`, `continent`, `channel`, `test`, `test_group`, `event_name`, `value`


In [4]:
required_columns = {
    "date",
    "country",
    "device",
    "continent",
    "channel",
    "test",
    "test_group",
    "event_name",
    "value"
}

missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise ValueError(
        f"Missing columns in the dataset: {sorted(missing_columns)}"
    )

df["value"] = pd.to_numeric(df["value"], errors="coerce").fillna(0)
df["test"] = pd.to_numeric(df["test"], errors="coerce")
df["test_group"] = pd.to_numeric(df["test_group"], errors="coerce")

print("Test numbers:", sorted(df["test"].dropna().unique()))
print("Test groups:", sorted(df["test_group"].dropna().unique()))
print("Events:")
display(
    df[["event_name"]]
    .drop_duplicates()
    .sort_values("event_name")
    .reset_index(drop=True)
)


Test numbers: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Test groups: [np.int64(1), np.int64(2)]
Events:


,event_name
0,add_payment_info
1,add_shipping_info
2,add_to_cart
3,begin_checkout
4,click
5,first_visit
6,new_account
7,page_view
8,scroll
9,select_item


## 4. Analysis settings

Based on the assignment example:

- `test_group = 1` is the control group;
- `test_group = 2` is the test group.

A **two-proportion z-test** is used to compare the two conversion rates.


In [5]:
ALPHA = 0.05

CONTROL_GROUP = 1
TEST_GROUP = 2

METRICS = [
    "add_payment_info",
    "add_shipping_info",
    "begin_checkout",
    "new_account"
]

SLICE_COLUMNS = [
    "country",
    "device",
    "continent",
    "channel"
]


## 5. Calculation functions

First, the data is transformed from long format into wide format.  
Then, the conversion rate in the test group is compared with the conversion rate in the control group for each metric.


In [6]:
def prepare_wide_table(data: pd.DataFrame, slice_columns=None) -> pd.DataFrame:
    if slice_columns is None:
        slice_columns = []

    index_columns = ["test", "test_group"] + slice_columns

    filtered_data = data[
        data["event_name"].isin(["session"] + METRICS)
    ].copy()

    wide = (
        filtered_data
        .groupby(index_columns + ["event_name"], dropna=False, as_index=False)["value"]
        .sum()
        .pivot_table(
            index=index_columns,
            columns="event_name",
            values="value",
            aggfunc="sum",
            fill_value=0
        )
        .reset_index()
    )

    for column in ["session"] + METRICS:
        if column not in wide.columns:
            wide[column] = 0

    return wide


def calculate_significance(data: pd.DataFrame, slice_columns=None) -> pd.DataFrame:
    if slice_columns is None:
        slice_columns = []

    wide = prepare_wide_table(data, slice_columns)
    group_columns = ["test"] + slice_columns
    results = []

    for keys, group_data in wide.groupby(group_columns, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)

        group_values = dict(zip(group_columns, keys))

        control_row = group_data[group_data["test_group"] == CONTROL_GROUP]
        test_row = group_data[group_data["test_group"] == TEST_GROUP]

        if control_row.empty or test_row.empty:
            continue

        control_row = control_row.iloc[0]
        test_row = test_row.iloc[0]

        control_sessions = int(control_row["session"])
        test_sessions = int(test_row["session"])

        if control_sessions == 0 or test_sessions == 0:
            continue

        for metric in METRICS:
            control_numerator = int(control_row[metric])
            test_numerator = int(test_row[metric])

            control_conversion = control_numerator / control_sessions
            test_conversion = test_numerator / test_sessions

            z_stat, p_value = proportions_ztest(
                count=[test_numerator, control_numerator],
                nobs=[test_sessions, control_sessions]
            )

            metric_change = (
                (test_conversion / control_conversion - 1) * 100
                if control_conversion != 0
                else np.nan
            )

            results.append({
                **group_values,
                "metric": metric,
                "numerator_test": test_numerator,
                "denominator_test": test_sessions,
                "conversion_rate_test": test_conversion,
                "numerator_control": control_numerator,
                "denominator_control": control_sessions,
                "conversion_rate_control": control_conversion,
                "metric_change_pct": metric_change,
                "z_stat": z_stat,
                "p_value": p_value,
                "significant": p_value < ALPHA
            })

    return pd.DataFrame(results)


## 6. Total results


In [7]:
total_results = calculate_significance(df)

if total_results.empty:
    raise ValueError("Unable to calculate the results. Check the dataset structure and the test_group values.")

total_results = (
    total_results
    .sort_values(["test", "metric"])
    .reset_index(drop=True)
)

display(total_results)


,test,metric,numerator_test,denominator_test,conversion_rate_test,numerator_control,denominator_control,conversion_rate_control,metric_change_pct,z_stat,p_value,significant
0,1,add_payment_info,983,45193,0.021751,964,45362,0.021251,2.352276,0.518551,0.604074,False
1,1,add_shipping_info,1992,45193,0.044078,1860,45362,0.041003,7.497264,2.291930,0.021910,True
2,1,begin_checkout,1992,45193,0.044078,1862,45362,0.041048,7.381800,2.258498,0.023915,True
3,1,new_account,3681,45193,0.081451,3823,45362,0.084278,-3.354299,-1.542883,0.122859,False
4,2,add_payment_info,1125,50244,0.022391,1096,50637,0.021644,3.448865,0.807895,0.419151,False
5,2,add_shipping_info,2136,50244,0.042513,2194,50637,0.043328,-1.882068,-0.638943,0.522860,False
6,2,begin_checkout,2137,50244,0.042532,2195,50637,0.043348,-1.880854,-0.638682,0.523030,False
7,2,new_account,4184,50244,0.083274,4165,50637,0.082252,1.241934,0.588793,0.556000,False
8,3,add_payment_info,1788,70439,0.025384,1771,70047,0.025283,0.398058,0.120029,0.904461,False
9,3,add_shipping_info,2889,70439,0.041014,2915,70047,0.041615,-1.443484,-0.565667,0.571620,False


### How to read the table

- `conversion_rate_test` is the conversion rate in the test group;
- `conversion_rate_control` is the conversion rate in the control group;
- `metric_change_pct` is the relative percentage change in the test group compared with the control group;
- `p_value` is the p-value;
- `significant = True` means that the difference is statistically significant at the `0.05` significance level.


## 7. Brief automatic summary for total results


In [8]:
def format_pct(value):
    return f"{value * 100:.2f}%"

for _, row in total_results.iterrows():
    direction = "increased" if row["metric_change_pct"] > 0 else "decreased"
    significance_text = (
        "The difference is statistically significant."
        if row["significant"]
        else "No statistically significant difference was detected."
    )

    print(
        f"Test {int(row['test'])}, metric {row['metric']}: "
        f"conversion rate {direction} by {abs(row['metric_change_pct']):.2f}%. "
        f"Control: {format_pct(row['conversion_rate_control'])}; "
        f"test: {format_pct(row['conversion_rate_test'])}; "
        f"p-value = {row['p_value']:.6f}. "
        f"{significance_text}"
    )


Test 1, metric add_payment_info: conversion rate increased by 2.35%. Control: 2.13%; test: 2.18%; p-value = 0.604074. No statistically significant difference was detected.
Test 1, metric add_shipping_info: conversion rate increased by 7.50%. Control: 4.10%; test: 4.41%; p-value = 0.021910. The difference is statistically significant.
Test 1, metric begin_checkout: conversion rate increased by 7.38%. Control: 4.10%; test: 4.41%; p-value = 0.023915. The difference is statistically significant.
Test 1, metric new_account: conversion rate decreased by 3.35%. Control: 8.43%; test: 8.15%; p-value = 0.122859. No statistically significant difference was detected.
Test 2, metric add_payment_info: conversion rate increased by 3.45%. Control: 2.16%; test: 2.24%; p-value = 0.419151. No statistically significant difference was detected.
Test 2, metric add_shipping_info: conversion rate decreased by 1.88%. Control: 4.33%; test: 4.25%; p-value = 0.522860. No statistically significant difference was d

## 8. Additional breakdown analysis

This section allows you to review the results separately by country, device, continent, and channel.

Because this section includes a large number of comparisons, an adjusted `p-value` is added using the Benjamini-Hochberg method.


In [9]:
slice_tables = []

for slice_column in SLICE_COLUMNS:
    current_slice = calculate_significance(
        df,
        slice_columns=[slice_column]
    )

    if current_slice.empty:
        continue

    current_slice.insert(1, "slice_type", slice_column)
    current_slice = current_slice.rename(
        columns={slice_column: "slice_value"}
    )

    slice_tables.append(current_slice)

if slice_tables:
    slice_results = pd.concat(slice_tables, ignore_index=True)

    slice_results["p_value_fdr"] = multipletests(
        slice_results["p_value"],
        method="fdr_bh"
    )[1]

    slice_results["significant_fdr"] = (
        slice_results["p_value_fdr"] < ALPHA
    )

    slice_results = (
        slice_results
        .sort_values(["test", "slice_type", "slice_value", "metric"])
        .reset_index(drop=True)
    )

    display(slice_results.head(50))
else:
    slice_results = pd.DataFrame()
    print("No data available for breakdown analysis.")


/usr/local/lib/python3.12/dist-packages/statsmodels/stats/weightstats.py:792: RuntimeWarning: invalid value encountered in scalar divide
  zstat = value / std
/usr/local/lib/python3.12/dist-packages/statsmodels/stats/weightstats.py:792: RuntimeWarning: invalid value encountered in scalar divide
  zstat = value / std
/usr/local/lib/python3.12/dist-packages/statsmodels/stats/weightstats.py:792: RuntimeWarning: invalid value encountered in scalar divide
  zstat = value / std
/usr/local/lib/python3.12/dist-packages/statsmodels/stats/weightstats.py:792: RuntimeWarning: invalid value encountered in scalar divide
  zstat = value / std
/usr/local/lib/python3.12/dist-packages/statsmodels/stats/weightstats.py:792: RuntimeWarning: invalid value encountered in scalar divide
  zstat = value / std
/usr/local/lib/python3.12/dist-packages/statsmodels/stats/weightstats.py:792: RuntimeWarning: invalid value encountered in scalar divide
  zstat = value / std
/usr/local/lib/python3.12/dist-packages/statsm

,test,slice_type,slice_value,metric,numerator_test,denominator_test,conversion_rate_test,numerator_control,denominator_control,conversion_rate_control,metric_change_pct,z_stat,p_value,significant,p_value_fdr,significant_fdr
0,1,channel,Direct,add_payment_info,228,10361,0.022006,205,10691,0.019175,14.761877,1.446625,0.148002,False,NaN,False
1,1,channel,Direct,add_shipping_info,438,10361,0.042274,419,10691,0.039192,7.864055,1.131323,0.257919,False,NaN,False
2,1,channel,Direct,begin_checkout,438,10361,0.042274,420,10691,0.039285,7.607236,1.096376,0.272914,False,NaN,False
3,1,channel,Direct,new_account,850,10361,0.082038,913,10691,0.085399,-3.935085,-0.879999,0.378860,False,NaN,False
4,1,channel,Organic Search,add_payment_info,250,15631,0.015994,291,15675,0.018565,-13.847516,-1.745184,0.080953,False,NaN,False
5,1,channel,Organic Search,add_shipping_info,648,15631,0.041456,629,15675,0.040128,3.310663,0.594160,0.552405,False,NaN,False
6,1,channel,Organic Search,begin_checkout,648,15631,0.041456,630,15675,0.040191,3.146677,0.565415,0.571791,False,NaN,False
7,1,channel,Organic Search,new_account,1301,15631,0.083232,1307,15675,0.083381,-0.178867,-0.047745,0.961919,False,NaN,False
8,1,channel,Paid Search,add_payment_info,236,11841,0.019931,225,11777,0.019105,4.321970,0.458639,0.646493,False,NaN,False
9,1,channel,Paid Search,add_shipping_info,498,11841,0.042057,485,11777,0.041182,2.125430,0.336760,0.736298,False,NaN,False


## 9. Statistically significant breakdown results after adjustment


In [10]:
if not slice_results.empty:
    significant_slices = slice_results[
        slice_results["significant_fdr"]
    ].copy()

    if significant_slices.empty:
        print("There are no statistically significant breakdown results after FDR adjustment.")
    else:
        display(significant_slices)
else:
    significant_slices = pd.DataFrame()
    print("No data available for breakdown analysis.")


There are no statistically significant breakdown results after FDR adjustment.


## 10. Export results to CSV


In [11]:
total_output_file = "ab_testing_total_results.csv"
slice_output_file = "ab_testing_slice_results.csv"
significant_slices_output_file = "ab_testing_significant_slices.csv"

total_results.to_csv(
    total_output_file,
    index=False
)

print(f"File generated: {total_output_file}")
files.download(total_output_file)

if not slice_results.empty:
    slice_results.to_csv(
        slice_output_file,
        index=False
    )

    print(f"File generated: {slice_output_file}")
    files.download(slice_output_file)

if not significant_slices.empty:
    significant_slices.to_csv(
        significant_slices_output_file,
        index=False
    )

    print(f"File generated: {significant_slices_output_file}")
    files.download(significant_slices_output_file)


File generated: ab_testing_total_results.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

File generated: ab_testing_slice_results.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>